In [1]:
import os
import json
import data
import random
import sklearn
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from sklearn.decomposition import NMF
from sklearn.utils import shuffle, check_random_state

import config
from utils import apply_symmetric_noise

random.seed(config.SEED)
np.random.seed(config.SEED)
np.random.default_rng(config.SEED)
check_random_state(config.SEED)

import warnings

warnings.filterwarnings("ignore")

In [2]:
DATA_SIZE = 1500
NOISE_PROB = 0.15
MAX_ITER = 10000
COLLECTION_NAME = "UTK"
DATA_FOLDER_PATH = Path(f"NMF_{COLLECTION_NAME}_{DATA_SIZE}_{str(NOISE_PROB).replace('.', '')}_denoise_info_test")

In [3]:
if COLLECTION_NAME == "UTK":
    array = shuffle(data.load_utk())[:DATA_SIZE]
elif COLLECTION_NAME == "Olivetti":
    array = shuffle(data.load_olivetti())[:DATA_SIZE]
elif COLLECTION_NAME == "Swimmer":
    array = shuffle(data.load_swimmer())[:DATA_SIZE]
else:
    raise ValueError

In [4]:
import numpy as np
import pandas as pd
from sklearn.decomposition import NMF

def _nmf_reconstruct(P, rank, seed=config.SEED, max_iter=5000):
    model = NMF(
        n_components=rank,
        max_iter=max_iter,
        random_state=seed,
        beta_loss="frobenius",
    )


    W = model.fit_transform(P) 
    H = model.components_ 
    return W @ H        

def _pop_mean_std(x: np.ndarray):
    m = x.mean()
    s = np.sqrt(((x - m) ** 2).mean()) 
    return m, s

def compute_table_vii_df(P: np.ndarray, ranks=(36, 72, 108), seed=1234, max_iter=5000) -> pd.DataFrame:
    """
    Implements Appendix A Eqs. (55)–(58) with population SDs.
    P must be (M, N), nonnegative (images as rows).
    Returns a DataFrame with rows indexed by R.
    """
    P = np.asarray(P, dtype=np.float64)
    if (P < 0).any():
        raise ValueError("P must be nonnegative for NMF.")
    if P.max() > 1.5:
        P = P / 255.0

    M, N = P.shape
    rows = []

    for R in ranks:
        P_hat = _nmf_reconstruct(P, rank=R, seed=seed, max_iter=max_iter)
        diff = P_hat - P

        num_i = np.abs(diff.sum(axis=1))               
        den_i = np.maximum(P_hat.sum(axis=1), P.sum(axis=1)) 
        n_i = np.divide(num_i, den_i, out=np.zeros_like(num_i), where=(den_i > 0))
        bar_n_I, sigma_n_I = _pop_mean_std(n_i)

        num_pi = np.abs(diff.sum(axis=0))                    
        den_pi = np.maximum(P_hat.sum(axis=0), P.sum(axis=0))
        n_pi = np.divide(num_pi, den_pi, out=np.zeros_like(num_pi), where=(den_pi > 0))
        bar_n_Pi, sigma_n_Pi = _pop_mean_std(n_pi)

        abs_diff = np.abs(diff)
        den_elem = np.maximum(P_hat, P)
        n_elem = np.divide(abs_diff, den_elem, out=np.zeros_like(abs_diff), where=(den_elem > 0))
        bar_n = n_elem.mean()
        sigma_n = np.sqrt(((n_elem - bar_n) ** 2).mean())

        rows.append({
            "R": R,
            "bar_nI": bar_n_I,
            "sigma(nI)": sigma_n_I,
            "bar_nΠ": bar_n_Pi,
            "sigma(nΠ)": sigma_n_Pi,
            "bar_n": bar_n,
            "sigma(n)": sigma_n,
        })

    df = pd.DataFrame(rows).set_index("R")
    return df.applymap(lambda x: float(x))


In [5]:
df = compute_table_vii_df(array, ranks=(36, 72, 108))

In [6]:
df

,bar_nI,sigma(nI),bar_nΠ,sigma(nΠ),bar_n,sigma(n)
R,,,,,,
36,0.002903,0.004275,0.002834,0.002353,0.124932,0.154559
72,0.002337,0.003328,0.002267,0.001755,0.102807,0.138685
108,0.002116,0.002912,0.002047,0.001512,0.090332,0.129257
